## Write a solution to find all the authors that viewed at least one of their own articles.

Return the result table sorted by id in ascending order.

The result format is in the following example.

 

Example 1:

Input: 
Views table:
+------------+-----------+-----------+------------+
| article_id | author_id | viewer_id | view_date  |
+------------+-----------+-----------+------------+
| 1          | 3         | 5         | 2019-08-01 |
| 1          | 3         | 6         | 2019-08-02 |
| 2          | 7         | 7         | 2019-08-01 |
| 2          | 7         | 6         | 2019-08-02 |
| 4          | 7         | 1         | 2019-07-22 |
| 3          | 4         | 4         | 2019-07-21 |
| 3          | 4         | 4         | 2019-07-21 |
+------------+-----------+-----------+------------+

## creating spark session 

In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Article View").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 05:38:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/27 05:38:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## importing pandas and taking data from leet code

In [2]:
import pandas as pd
data = [[1, 3, 5, '2019-08-01'], [1, 3, 6, '2019-08-02'], [2, 7, 7, '2019-08-01'], [2, 7, 6, '2019-08-02'], [4, 7, 1, '2019-07-22'], [3, 4, 4, '2019-07-21'], [3, 4, 4, '2019-07-21']]
views = pd.DataFrame(data, columns=['article_id', 'author_id', 'viewer_id', 'view_date']).astype({'article_id':'Int64', 'author_id':'Int64', 'viewer_id':'Int64', 'view_date':'datetime64[ns]'})

## converting pandas data frame to spark Data Frame

In [9]:
df=spark.createDataFrame(views)
df.show()

+----------+---------+---------+-------------------+
|article_id|author_id|viewer_id|          view_date|
+----------+---------+---------+-------------------+
|         1|        3|        5|2019-08-01 00:00:00|
|         1|        3|        6|2019-08-02 00:00:00|
|         2|        7|        7|2019-08-01 00:00:00|
|         2|        7|        6|2019-08-02 00:00:00|
|         4|        7|        1|2019-07-22 00:00:00|
|         3|        4|        4|2019-07-21 00:00:00|
|         3|        4|        4|2019-07-21 00:00:00|
+----------+---------+---------+-------------------+



In [7]:
from pyspark.sql.functions import col
df1= df.withColumn("id",col("author_id")).filter("author_id=viewer_id").distinct().select("id").orderBy("id",ascending=True)
df1.show()

+---+
| id|
+---+
|  4|
|  7|
+---+



## creating temp View table of DataFrame

In [10]:
df.createTempView("views")

AnalysisException: [TEMP_TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create the temporary view `views` because it already exists.
Choose a different name, drop or replace the existing view,  or add the IF NOT EXISTS clause to tolerate pre-existing views.

In [12]:
df2=spark.sql("""
          select
               author_id as id
          from views
          where author_id=viewer_id
          order by author_id""")
df2.show()

+---+
| id|
+---+
|  4|
|  4|
|  7|
+---+

